In [ ]:
"""
结构探针实验脚本

功能：
1. 用模型对 benchmark 数据进行推理
2. 找到模型输出表格结构中（自回归角度）第一个出错的地方
3. 修正第一个出错处后，再次传入模型推理后续部分
4. 对比修正前后的：出错概率、正确/错误结构的生成概率 logp

用法：
    python structure_probe_experiment.py \
        --model_path tencent/HunyuanOCR \
        --input_file /path/to/benchmark.jsonl \
        --output_file /path/to/output.jsonl
"""

import argparse
import json
import os
import re
from typing import Any, Callable, Optional

from bs4 import BeautifulSoup, NavigableString
from PIL import Image
from transformers import AutoProcessor, ProcessorMixin
from vllm import LLM, SamplingParams

DEFAULT_MODEL_PATH = "tencent/HunyuanOCR"
GPU_UTILS = 0.9
INPUT_FILE = "your/path/to/table/parsing/benchmark.jsonl"
OUTPUT_FILE = INPUT_FILE.replace(".jsonl", "_structure_probe.jsonl")
DEFAULT_QUESTION = "把图中的表格解析为 HTML。"

In [ ]:
parser = argparse.ArgumentParser(description="结构探针实验：分析修正模型第一个结构错误对后续生成的影响")
parser.add_argument("--model_path", type=str, default=DEFAULT_MODEL_PATH)
parser.add_argument("--input_file", type=str, default=INPUT_FILE, help="Benchmark 数据文件路径 (JSONL 格式)")
parser.add_argument("--output_file", type=str, default=OUTPUT_FILE, help="输出结果文件路径 (JSONL 格式)")
parser.add_argument("--max_samples", type=int, default=-1, help="最大处理样本数 (-1 表示全部处理)")
parser.add_argument("--max_tokens", type=int, default=16384, help="最大生成 token 数 (默认: 16384)")
parser.add_argument("--gpu_memory_utilization", type=float, default=GPU_UTILS, help="GPU 显存利用率")
parser.add_argument("--tensor_parallel_size", type=int, default=1, help="张量并行大小 (默认: 1)")

args, _ = parser.parse_known_args()

print("=" * 80)
print("结构探针实验")
print("=" * 80)
print(f"  模型路径: {args.model_path}")
print(f"  输入文件: {args.input_file}")
print(f"  输出文件: {args.output_file}")
print(f"  最大样本数: {'全部' if args.max_samples == -1 else args.max_samples}")
print(f"  最大生成 token 数: {args.max_tokens}")
print("=" * 80)

# ---- 加载数据 ----
print("\n📂 加载 benchmark 数据...")
samples = []
with open(args.input_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

if args.max_samples > 0:
    samples = samples[: args.max_samples]

print(f"  共加载 {len(samples)} 条数据")

# ---- 初始化模型 ----
print("\n🚀 初始化模型...")
llm = LLM(
    model=args.model_path,
    trust_remote_code=True,
    gpu_memory_utilization=args.gpu_memory_utilization,
    tensor_parallel_size=args.tensor_parallel_size,
)
processor: ProcessorMixin = AutoProcessor.from_pretrained(args.model_path)
print("  模型加载完成")

In [ ]:
# ============================================================
# 第一部分：表格结构清理与比较工具
# ============================================================


def clean_table_content(html_str: str) -> str:
    """
    使用 BeautifulSoup 清空所有 td/th 内部内容，
    完全保留结构和属性。
    """
    soup = BeautifulSoup(html_str, "html.parser")

    # 清空所有单元格内容
    for cell in soup.find_all(["td", "th"]):
        cell.clear()

    # 清理 <tr> 中的 stray text
    for tr in soup.find_all("tr"):
        for content in list(tr.contents):
            if isinstance(content, NavigableString):
                content.extract()

    return str(soup)


def extract_cells_with_row_info(soup: BeautifulSoup) -> list[tuple]:
    """
    提取所有单元格及其所在的行索引

    Returns:
        list of tuples: [(cell, row_index), ...]
    """
    cells_with_row = []
    rows = soup.find_all("tr")

    for row_idx, row in enumerate(rows):
        cells = row.find_all(["td", "th"])
        for cell in cells:
            cells_with_row.append((cell, row_idx))

    return cells_with_row


def compare_cell_structure(
    cell1: BeautifulSoup,
    row_idx1: int,
    cell2: BeautifulSoup,
    row_idx2: int,
) -> bool:
    """
    比较两个单元格的结构是否相同（包括所在行）
    """
    if row_idx1 != row_idx2:
        return False
    if cell1.name != cell2.name:
        return False
    if dict(cell1.attrs) != dict(cell2.attrs):
        return False
    return True


def find_first_structure_error(response_structure: str, ref_structure: str) -> dict:
    """
    找到模型输出结构中第一个与参考答案不同的 cell 位置。

    Returns:
        dict: {
            "first_error_index": int,   # 第一个出错的 cell index（-1 表示完全匹配或无法比较）
            "matched_cells": int,       # 匹配的 cell 数
            "total_ref_cells": int,     # 参考答案总 cell 数
            "total_resp_cells": int,    # 模型输出总 cell 数
            "ref_cells": list,          # 参考答案的 cell 列表
            "resp_cells": list,         # 模型输出的 cell 列表
        }
    """
    soup_response = BeautifulSoup(response_structure, "html.parser")
    soup_ref = BeautifulSoup(ref_structure, "html.parser")

    cells_response = extract_cells_with_row_info(soup_response)
    cells_ref = extract_cells_with_row_info(soup_ref)

    ref_cell_count = len(cells_ref)
    resp_cell_count = len(cells_response)

    if response_structure == ref_structure:
        return {
            "first_error_index": -1,
            "matched_cells": ref_cell_count,
            "total_ref_cells": ref_cell_count,
            "total_resp_cells": resp_cell_count,
            "ref_cells": cells_ref,
            "resp_cells": cells_response,
        }

    matched_cells = 0
    for i in range(min(ref_cell_count, resp_cell_count)):
        cell_ref, row_idx_ref = cells_ref[i]
        cell_resp, row_idx_resp = cells_response[i]

        if not compare_cell_structure(cell_ref, row_idx_ref, cell_resp, row_idx_resp):
            break
        matched_cells += 1

    return {
        "first_error_index": matched_cells,
        "matched_cells": matched_cells,
        "total_ref_cells": ref_cell_count,
        "total_resp_cells": resp_cell_count,
        "ref_cells": cells_ref,
        "resp_cells": cells_response,
    }


# ============================================================
# 第二部分：token 级别的结构定位与分割工具
# ============================================================


def tokenize_html_structure(html_str: str) -> list[str]:
    """
    将 HTML 结构字符串按 token 粒度拆分，
    方便后续在原始（含内容）的 HTML 上做对齐。
    这里是粗粒度的：按标签和文本交替切分。
    """
    # 按 HTML 标签分割
    parts = re.split(r"(<[^>]+>)", html_str)
    return [p for p in parts if p]


def rebuild_structure_prefix_from_ref(
    ref_html: str,
    ref_structure: str,
    num_matched_cells: int,
) -> str:
    """
    从参考答案中，提取前 num_matched_cells + 1 个 cell 对应的 HTML 前缀（含内容）。
    这里我们返回的是参考答案中到第 (num_matched_cells+1) 个 cell 结束为止的 HTML。

    用于构造修正后的 prefix：
    - 前 num_matched_cells 个 cell 已经匹配
    - 第 (num_matched_cells+1) 个 cell 是被修正的 cell（用参考答案替换）

    Args:
        ref_html: 参考答案的完整 HTML（含内容）
        ref_structure: 参考答案的结构 HTML（已清空内容）
        num_matched_cells: 已匹配的 cell 数

    Returns:
        str: 参考答案中到第 (num_matched_cells+1) 个 cell 结束为止的 HTML 前缀
    """
    soup = BeautifulSoup(ref_html, "html.parser")
    all_cells = soup.find_all(["td", "th"])

    # 我们需要前 num_matched_cells + 1 个 cell
    target_count = num_matched_cells + 1
    if target_count > len(all_cells):
        # 如果参考答案的 cell 数量不够，返回整个参考答案
        return ref_html

    # 找到第 target_count 个 cell 的结束位置
    # 使用 sourceline 和 sourcepos 来定位在原始 HTML 中的位置
    # 更可靠的方法：直接在原始字符串上做搜索

    # 逐个查找 cell 的结束标签位置
    cell_tag_pattern = re.compile(r"<(td|th)\b[^>]*>.*?</\1>", re.DOTALL)
    cell_end_pos = 0
    count = 0

    for match in cell_tag_pattern.finditer(ref_html):
        count += 1
        cell_end_pos = match.end()
        if count >= target_count:
            break

    if count < target_count:
        return ref_html

    return ref_html[:cell_end_pos]


def find_cell_boundary_in_text(text: str, cell_index: int) -> tuple[int, int]:
    """
    在原始 HTML 文本中找到第 cell_index 个 cell (td/th) 的开始和结束位置。
    cell_index 从 0 开始。

    Returns:
        (start_pos, end_pos): cell 的开始和结束位置在文本中的字符索引
    """
    # 匹配完整的 td/th 标签（包括内容和关闭标签）
    cell_pattern = re.compile(r"<(td|th)\b[^>]*>.*?</\1>", re.DOTALL)

    count = 0
    for match in cell_pattern.finditer(text):
        if count == cell_index:
            return match.start(), match.end()
        count += 1

    return -1, -1


# ============================================================
# 第三部分：vLLM 推理相关工具
# ============================================================


def build_initial_prompt(
    processor: ProcessorMixin,
    img_path: str,
    question: str,
) -> tuple[str, Image.Image]:
    """
    构建初始推理的 prompt 和图片数据。

    Returns:
        (prompt_str, pil_image)
    """
    img = Image.open(img_path)
    messages = [
        {"role": "system", "content": ""},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": question},
            ],
        },
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt, img


def build_corrected_prompt(
    processor: ProcessorMixin,
    question: str,
    corrected_prefix: str,
    prompt_modifier: Optional[Callable[[str], str]] = None,
) -> str:
    """
    构建修正后的 prompt（将修正后的前缀作为 assistant 的部分内容）。

    Args:
        processor: tokenizer/processor
        question: 原始问题
        corrected_prefix: 修正后的前缀文本（用参考答案替换了出错的 cell）
        prompt_modifier: 可选的 prompt 后处理函数

    Returns:
        prompt_str: 包含修正前缀的 prompt
    """
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        },
    ]

    if corrected_prefix:
        conversation.append(
            {
                "role": "assistant",
                "content": [{"type": "text", "text": corrected_prefix}],
            }
        )

    add_gen_prompt = len(conversation) == 1
    prompt = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=add_gen_prompt,
    )

    if corrected_prefix and prompt_modifier is not None:
        prompt = prompt_modifier(prompt)

    return prompt


def run_inference_with_logprobs(
    llm: LLM,
    prompt: str,
    image: Image.Image,
    max_tokens: int = 16384,
) -> dict:
    """
    运行推理并获取 logprobs。

    Returns:
        dict: {
            "text": str,           # 生成的文本
            "logprobs": list,      # 每个 token 的 logprob
            "tokens": list,        # 生成的 token 列表
        }
    """
    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=max_tokens,
        logprobs=1,  # 返回 top-1 的 logprob
    )

    inputs = {
        "prompt": prompt,
        "multi_modal_data": {"image": [image]},
    }

    output = llm.generate([inputs], sampling_params)[0]
    generated_output = output.outputs[0]

    text = generated_output.text
    tokens = []
    logprobs_list = []

    if generated_output.logprobs:
        for lp_dict in generated_output.logprobs:
            # lp_dict 是 {token_id: Logprob} 的字典
            for token_id, logprob_obj in lp_dict.items():
                tokens.append(logprob_obj.decoded_token)
                logprobs_list.append(logprob_obj.logprob)
                break  # 只取 top-1

    return {
        "text": text,
        "logprobs": logprobs_list,
        "tokens": tokens,
    }


# ============================================================
# 第四部分：分析与统计工具
# ============================================================


def analyze_logprobs_after_error(
    original_result: dict,
    corrected_result: dict,
    original_full_text: str,
    corrected_prefix: str,
    ref_html: str,
    first_error_cell_index: int,
) -> dict:
    """
    分析修正前后，出错位置之后的生成质量。

    对比指标：
    1. 出错概率：在出错位置之后，结构出错的 cell 比例
    2. 正确结构部分的平均 logp
    3. 错误结构部分的平均 logp

    Args:
        original_result: 原始推理结果 {"text", "logprobs", "tokens"}
        corrected_result: 修正后推理结果 {"text", "logprobs", "tokens"}
        original_full_text: 原始完整输出文本
        corrected_prefix: 修正用的前缀
        ref_html: 参考答案的完整 HTML
        first_error_cell_index: 第一个出错的 cell index

    Returns:
        dict: 各项分析指标
    """
    # 对原始输出做结构分析
    ref_structure = clean_table_content(ref_html)

    # --- 分析原始输出 ---
    orig_text = original_result["text"]
    orig_structure = clean_table_content(orig_text)
    orig_error_info = find_first_structure_error(orig_structure, ref_structure)

    # 分析原始输出中，出错之后的区域
    orig_after_error_analysis = _analyze_after_position(
        original_result, orig_text, ref_html, ref_structure, first_error_cell_index
    )

    # --- 分析修正后输出 ---
    # 修正后的完整输出 = corrected_prefix + corrected_result["text"]
    corrected_full_text = corrected_prefix + corrected_result["text"]
    corrected_structure = clean_table_content(corrected_full_text)
    corrected_error_info = find_first_structure_error(corrected_structure, ref_structure)

    # 修正后的分析：从修正点之后（即 first_error_cell_index + 1）开始
    corrected_after_error_analysis = _analyze_after_position(
        corrected_result, corrected_full_text, ref_html, ref_structure, first_error_cell_index + 1
    )

    return {
        "original": {
            "full_text": orig_text,
            "structure": orig_structure,
            "first_error_in_original": orig_error_info["first_error_index"],
            "matched_cells": orig_error_info["matched_cells"],
            "total_ref_cells": orig_error_info["total_ref_cells"],
            "total_resp_cells": orig_error_info["total_resp_cells"],
            "after_error_analysis": orig_after_error_analysis,
        },
        "corrected": {
            "full_text": corrected_full_text,
            "generated_text": corrected_result["text"],
            "structure": corrected_structure,
            "first_error_in_corrected": corrected_error_info["first_error_index"],
            "matched_cells": corrected_error_info["matched_cells"],
            "total_ref_cells": corrected_error_info["total_ref_cells"],
            "total_resp_cells": corrected_error_info["total_resp_cells"],
            "after_error_analysis": corrected_after_error_analysis,
        },
    }


def _analyze_after_position(
    inference_result: dict,
    full_text: str,
    ref_html: str,
    ref_structure: str,
    start_cell_index: int,
) -> dict:
    """
    分析从 start_cell_index 开始之后的结构正确性和 logp。

    Returns:
        dict: {
            "error_rate": float,         # 出错的 cell 比例
            "correct_cells": int,        # 正确的 cell 数
            "error_cells": int,          # 错误的 cell 数
            "total_cells_after": int,    # 出错后的总 cell 数
            "avg_logp": float,           # 平均 logp
            "correct_structure_avg_logp": float,   # 正确结构部分的平均 logp
            "error_structure_avg_logp": float,     # 错误结构部分的平均 logp
        }
    """
    full_structure = clean_table_content(full_text)

    soup_resp = BeautifulSoup(full_structure, "html.parser")
    soup_ref = BeautifulSoup(ref_structure, "html.parser")

    cells_resp = extract_cells_with_row_info(soup_resp)
    cells_ref = extract_cells_with_row_info(soup_ref)

    ref_count = len(cells_ref)
    resp_count = len(cells_resp)

    # 统计 start_cell_index 之后的结构正确性
    correct_cells = 0
    error_cells = 0

    for i in range(start_cell_index, min(ref_count, resp_count)):
        cell_ref, row_ref = cells_ref[i]
        cell_resp, row_resp = cells_resp[i]
        if compare_cell_structure(cell_ref, row_ref, cell_resp, row_resp):
            correct_cells += 1
        else:
            error_cells += 1

    # 如果模型输出的 cell 数和参考不一致，多出或缺少的也算错误
    if resp_count > ref_count:
        error_cells += resp_count - max(ref_count, start_cell_index)
    elif resp_count < ref_count:
        error_cells += ref_count - max(resp_count, start_cell_index)

    total_after = max(ref_count - start_cell_index, 0)
    error_rate = error_cells / total_after if total_after > 0 else 0.0

    # 计算 logp 相关
    # 尝试找到 start_cell_index 对应的 token 位置
    tokens = inference_result.get("tokens", [])
    logprobs = inference_result.get("logprobs", [])

    avg_logp = 0.0
    correct_structure_avg_logp = 0.0
    error_structure_avg_logp = 0.0

    if tokens and logprobs:
        # 将 tokens 合并看看在全文中的位置
        # 计算整体平均 logp
        if logprobs:
            avg_logp = sum(logprobs) / len(logprobs)

        # 尝试按结构标签分段统计 logp
        # 简化做法：将结构标签的 logp 统计出来
        structure_tag_logps = _extract_structure_tag_logps(tokens, logprobs)

        correct_lps = []
        error_lps = []

        # 按 cell 分组的简单逻辑：
        # 每次遇到 <td 或 <th 就算一个新的 cell 开始
        cell_logp_groups = _group_logps_by_cells(tokens, logprobs)

        for i, cell_lps in enumerate(cell_logp_groups):
            global_cell_idx = i  # cell_logp_groups 从 inference_result 的第一个 token 开始算
            # 跳过 start_cell_index 之前的
            if global_cell_idx < start_cell_index:
                continue
            if global_cell_idx >= min(ref_count, resp_count):
                break

            cell_ref, row_ref = cells_ref[global_cell_idx] if global_cell_idx < ref_count else (None, -1)
            cell_resp, row_resp = cells_resp[global_cell_idx] if global_cell_idx < resp_count else (None, -1)

            cell_avg_lp = sum(cell_lps) / len(cell_lps) if cell_lps else 0.0

            if cell_ref is not None and cell_resp is not None:
                if compare_cell_structure(cell_ref, row_ref, cell_resp, row_resp):
                    correct_lps.append(cell_avg_lp)
                else:
                    error_lps.append(cell_avg_lp)

        correct_structure_avg_logp = sum(correct_lps) / len(correct_lps) if correct_lps else 0.0
        error_structure_avg_logp = sum(error_lps) / len(error_lps) if error_lps else 0.0

    return {
        "error_rate": error_rate,
        "correct_cells": correct_cells,
        "error_cells": error_cells,
        "total_cells_after": total_after,
        "avg_logp": avg_logp,
        "correct_structure_avg_logp": correct_structure_avg_logp,
        "error_structure_avg_logp": error_structure_avg_logp,
    }


def _extract_structure_tag_logps(tokens: list[str], logprobs: list[float]) -> list[float]:
    """
    提取结构标签（td, th, tr, table, tbody, thead 等）的 logp。
    """
    structure_tags = {
        "<td",
        "<th",
        "</td>",
        "</th>",
        "<tr>",
        "</tr>",
        "<table>",
        "</table>",
        "<tbody>",
        "</tbody>",
        "<thead>",
        "</thead>",
    }
    result = []
    for tok, lp in zip(tokens, logprobs):
        tok_stripped = tok.strip()
        if any(tok_stripped.startswith(tag) for tag in structure_tags):
            result.append(lp)
    return result


def _group_logps_by_cells(tokens: list[str], logprobs: list[float]) -> list[list[float]]:
    """
    按 cell (td/th) 将 logps 分组。
    每次遇到 <td 或 <th 标签开始，认为是一个新 cell 的开始。

    Returns:
        list[list[float]]: 每个 cell 对应的 logprobs 列表
    """
    groups = []
    current_group = []
    in_cell = False

    for tok, lp in zip(tokens, logprobs):
        tok_stripped = tok.strip()

        # 检测新 cell 的开始
        if tok_stripped.startswith("<td") or tok_stripped.startswith("<th"):
            if current_group and in_cell:
                groups.append(current_group)
            current_group = [lp]
            in_cell = True
        elif in_cell:
            current_group.append(lp)

            # 检测 cell 的结束
            if tok_stripped == "</td>" or tok_stripped == "</th>":
                groups.append(current_group)
                current_group = []
                in_cell = False

    # 如果还有未完成的 group
    if current_group and in_cell:
        groups.append(current_group)

    return groups


# ============================================================
# 第五部分：修正前缀构建工具
# ============================================================


def build_corrected_prefix_from_original(
    original_text: str,
    ref_html: str,
    first_error_cell_index: int,
) -> str:
    """
    构建修正后的前缀：
    - 保留原始输出中前 first_error_cell_index 个 cell 的内容
    - 将第 first_error_cell_index 个 cell 用参考答案的对应 cell 替换

    Args:
        original_text: 模型原始输出文本
        ref_html: 参考答案 HTML
        first_error_cell_index: 第一个出错的 cell 索引，从 0 开始

    Returns:
        str: 修正后的前缀文本
    """
    if first_error_cell_index < 0:
        return original_text

    # 找到原始输出中第 first_error_cell_index - 1 个 cell 的边界
    orig_start, orig_end = find_cell_boundary_in_text(original_text, first_error_cell_index)

    # 找到参考答案中第 first_error_cell_index 个 cell 的边界
    ref_start, ref_end = find_cell_boundary_in_text(ref_html, first_error_cell_index)

    if orig_start == -1:
        return rebuild_structure_prefix_from_ref(ref_html, clean_table_content(ref_html), first_error_cell_index)
    elif ref_start == -1:
        ori = original_text[:orig_start]
        if ori.endswith("<tr>"):
            ori = ori[:-4]
        corrected_prefix = ori + "</table>"
    else:
        # 构建修正前缀：原始输出的前半部分 + 参考答案的修正 cell
        corrected_prefix = original_text[:orig_start] + ref_html[ref_start:ref_end]

    return corrected_prefix


# ============================================================
# 第六部分：prompt 后处理（移除 eos 等）
# ============================================================


def remove_trailing_eos(prompt: str) -> str:
    """
    移除 prompt 末尾的 eos token，使模型可以继续生成。
    不同模型的 eos token 可能不同，这里处理常见的情况。
    """
    # 常见的 eos token 模式
    eos_patterns = [
        "<|im_end|>",
        "</s>",
        "<|endoftext|>",
        "<|eot_id|>",
    ]

    for pattern in eos_patterns:
        if prompt.endswith(pattern):
            prompt = prompt[: -len(pattern)]

    return prompt


# ============================================================
# 第七部分：主流程
# ============================================================


def process_single_sample(
    llm: LLM,
    processor: Any,
    sample: dict,
    sample_index: int,
) -> dict:
    """
    处理单条 benchmark 数据。

    Args:
        llm: vLLM 的 LLM 实例
        processor: AutoProcessor 实例
        sample: 单条数据 dict
        sample_index: 数据索引

    Returns:
        dict: 包含分析结果的字典
    """
    image_path = sample["image_path"][0]
    # question = sample["问题"][0]
    question = DEFAULT_QUESTION
    ref_html = sample["参考答案"][0]

    print(f"\n{'=' * 80}")
    print(f"处理样本 #{sample_index}, ID: {sample.get('id', 'N/A')}")
    print(f"图片路径: {image_path}")

    # 检查图片是否存在
    if not os.path.exists(image_path):
        print(f"  ⚠️  图片不存在，跳过: {image_path}")
        return {
            "id": sample.get("id", "N/A"),
            "image_path": image_path,
            "status": "skipped",
            "reason": "image_not_found",
        }

    # ---- Step 1: 原始推理 ----
    print("  Step 1: 原始推理...")
    prompt, img = build_initial_prompt(processor, image_path, question)
    print(f"  prompt: {prompt}")
    original_result = run_inference_with_logprobs(llm, prompt, img)
    original_text = original_result["text"]
    print(f"  原始输出: {original_text[:100]}...")

    # ---- Step 2: 结构比较，找第一个错误 ----
    print("  Step 2: 结构比较...")
    ref_structure = clean_table_content(ref_html)
    orig_structure = clean_table_content(original_text)
    error_info = find_first_structure_error(orig_structure, ref_structure)

    first_error_idx = error_info["first_error_index"]
    matched = error_info["matched_cells"]
    total_ref = error_info["total_ref_cells"]
    total_resp = error_info["total_resp_cells"]

    print(f"  参考答案 cell 数: {total_ref}, 模型输出 cell 数: {total_resp}")
    print(f"  匹配 cell 数: {matched}, 第一个出错位置: {first_error_idx}")

    if first_error_idx == -1:
        # 完全匹配，不需要修正
        print("  ✅ 结构完全匹配，无需修正")

        # 仍然计算 logprob 统计
        avg_logp = 0.0
        if original_result["logprobs"]:
            avg_logp = sum(original_result["logprobs"]) / len(original_result["logprobs"])

        return {
            "id": sample.get("id", "N/A"),
            "image_path": image_path,
            "status": "perfect_match",
            "matched_cells": matched,
            "total_ref_cells": total_ref,
            "total_resp_cells": total_resp,
            "original_avg_logp": avg_logp,
            "original_text": original_text,
        }

    # ---- Step 3: 构建修正前缀 ----
    print("  Step 3: 构建修正前缀...")
    corrected_prefix = build_corrected_prefix_from_original(original_text, ref_html, first_error_idx)
    print(f"  修正前缀长度: {len(corrected_prefix)} 字符")

    # ---- Step 4: 使用修正前缀进行二次推理 ----
    print("  Step 4: 修正后推理...")
    corrected_prompt = prompt + corrected_prefix
    if corrected_prompt.endswith(">"):
        corrected_prompt = corrected_prompt[:-1]
    print(f"  修正后 prompt: {corrected_prompt}")
    corrected_result = run_inference_with_logprobs(llm, corrected_prompt, img)
    print(f"  修正后生成: {corrected_result['text'][:100]}...")

    # ---- Step 5: 对比分析 ----
    print("  Step 5: 对比分析...")
    analysis = analyze_logprobs_after_error(
        original_result=original_result,
        corrected_result=corrected_result,
        original_full_text=original_text,
        corrected_prefix=corrected_prefix,
        ref_html=ref_html,
        first_error_cell_index=first_error_idx,
    )

    # 构建输出结果
    result = {
        "id": sample.get("id", "N/A"),
        "image_path": image_path,
        "status": "analyzed",
        "first_error_cell_index": first_error_idx,
        "matched_cells": matched,
        "total_ref_cells": total_ref,
        "total_resp_cells": total_resp,
        # 原始输出分析
        "original": {
            "text": original_text,
            "structure_score": matched / total_ref if total_ref > 0 else 0.0,
            "after_error": analysis["original"]["after_error_analysis"],
        },
        # 修正后输出分析
        "corrected": {
            "prefix": corrected_prefix,
            "generated_text": analysis["corrected"]["generated_text"],
            "full_text": analysis["corrected"]["full_text"],
            "structure_score": analysis["corrected"]["matched_cells"] / total_ref if total_ref > 0 else 0.0,
            "after_error": analysis["corrected"]["after_error_analysis"],
        },
        # 关键对比指标
        "comparison": {
            "original_error_rate_after": analysis["original"]["after_error_analysis"]["error_rate"],
            "corrected_error_rate_after": analysis["corrected"]["after_error_analysis"]["error_rate"],
            "original_correct_logp": analysis["original"]["after_error_analysis"]["correct_structure_avg_logp"],
            "corrected_correct_logp": analysis["corrected"]["after_error_analysis"]["correct_structure_avg_logp"],
            "original_error_logp": analysis["original"]["after_error_analysis"]["error_structure_avg_logp"],
            "corrected_error_logp": analysis["corrected"]["after_error_analysis"]["error_structure_avg_logp"],
            "original_avg_logp": analysis["original"]["after_error_analysis"]["avg_logp"],
            "corrected_avg_logp": analysis["corrected"]["after_error_analysis"]["avg_logp"],
        },
    }

    # 打印关键对比指标
    comp = result["comparison"]
    print("\n  📊 对比结果:")
    print(
        f"    出错率:  原始 {comp['original_error_rate_after']:.4f}  →  修正后 {comp['corrected_error_rate_after']:.4f}"
    )
    print(
        f"    正确结构 logp:  原始 {comp['original_correct_logp']:.4f}  →  修正后 {comp['corrected_correct_logp']:.4f}"
    )
    print(f"    错误结构 logp:  原始 {comp['original_error_logp']:.4f}  →  修正后 {comp['corrected_error_logp']:.4f}")
    print(f"    整体平均 logp:  原始 {comp['original_avg_logp']:.4f}  →  修正后 {comp['corrected_avg_logp']:.4f}")

    return result


# ---- 逐条处理 ----
print("\n🔬 开始实验...")
os.makedirs(os.path.dirname(os.path.abspath(args.output_file)), exist_ok=True)

# 统计变量
total = 0
perfect_match = 0
improved = 0
degraded = 0
skipped = 0

with open(args.output_file, "w", encoding="utf-8") as fout:
    for idx, sample in enumerate(samples):
        total += 1

        result = process_single_sample(llm, processor, sample, idx)

        # 统计
        if result["status"] == "skipped":
            skipped += 1
        elif result["status"] == "perfect_match":
            perfect_match += 1
        elif result["status"] == "analyzed":
            comp = result["comparison"]
            if comp["corrected_error_rate_after"] < comp["original_error_rate_after"]:
                improved += 1
            elif comp["corrected_error_rate_after"] > comp["original_error_rate_after"]:
                degraded += 1

        # 写入结果
        fout.write(json.dumps(result, ensure_ascii=False) + "\n")
        fout.flush()

# ---- 汇总报告 ----
print("\n" + "=" * 80)
print("📋 实验汇总报告")
print("=" * 80)
print(f"  总样本数: {total}")
print(f"  跳过: {skipped}")
print(f"  完全匹配: {perfect_match}")
analyzed = total - skipped - perfect_match
print(f"  有结构错误（已分析）: {analyzed}")
if analyzed > 0:
    print(f"    修正后改善: {improved} ({improved / analyzed * 100:.1f}%)")
    print(f"    修正后恶化: {degraded} ({degraded / analyzed * 100:.1f}%)")
    print(
        f"    修正后无变化: {analyzed - improved - degraded} ({(analyzed - improved - degraded) / analyzed * 100:.1f}%)"
    )
print(f"\n  结果已保存至: {args.output_file}")
print("=" * 80)

# ---- 计算整体平均指标 ----
if analyzed > 0:
    print("\n📊 整体平均对比指标:")
    avg_metrics = {
        "original_error_rate": [],
        "corrected_error_rate": [],
        "original_correct_logp": [],
        "corrected_correct_logp": [],
        "original_error_logp": [],
        "corrected_error_logp": [],
        "original_avg_logp": [],
        "corrected_avg_logp": [],
    }

    # 重新读取结果文件统计
    with open(args.output_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                if r.get("status") == "analyzed":
                    comp = r["comparison"]
                    avg_metrics["original_error_rate"].append(comp["original_error_rate_after"])
                    avg_metrics["corrected_error_rate"].append(comp["corrected_error_rate_after"])
                    avg_metrics["original_correct_logp"].append(comp["original_correct_logp"])
                    avg_metrics["corrected_correct_logp"].append(comp["corrected_correct_logp"])
                    avg_metrics["original_error_logp"].append(comp["original_error_logp"])
                    avg_metrics["corrected_error_logp"].append(comp["corrected_error_logp"])
                    avg_metrics["original_avg_logp"].append(comp["original_avg_logp"])
                    avg_metrics["corrected_avg_logp"].append(comp["corrected_avg_logp"])

    for key, values in avg_metrics.items():
        if values:
            avg = sum(values) / len(values)
            print(f"  {key}: {avg:.6f}")

print("\n✅ 实验完成!")
